# Agent Comparison

This notebook compares the current Hanabi benchmark agents using the same
report builder as `hanabi-evaluate`, so the interactive view stays aligned
with the CLI benchmark.

Included agents:

- `RandomAgent`
- `BasicHeuristicAgent`
- `ConventionHeuristicAgent`
- `TempoHeuristicAgent`
- `ConventionTempoHeuristicAgent`


In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

try:
    import hanabi_ai  # noqa: F401
    print("Using installed hanabi_ai package.")
except ModuleNotFoundError:
    if str(SRC_PATH) not in sys.path:
        sys.path.insert(0, str(SRC_PATH))
    print("hanabi_ai not installed; using src/ fallback.")

print(f"Project root: {PROJECT_ROOT}")


Using installed hanabi_ai package.
Project root: c:\Users\marce\Documents\GitHub\hanabi-ai


In [12]:
from hanabi_ai.tools.evaluate_agents import build_benchmark_report


In [13]:
# Edit these values when you want to record a new comparison run.
PLAYER_COUNTS = [2, 3, 4, 5]
GAME_COUNT = 200
SEED_BASE = 0
AGENT_SEED_BASE = 1000


In [14]:
report = build_benchmark_report(
    player_counts=PLAYER_COUNTS,
    game_count=GAME_COUNT,
    seed_base=SEED_BASE,
    agent_seed_base=AGENT_SEED_BASE,
)

report["config"]


KeyboardInterrupt: 

In [ ]:
def build_rows(result_set):
    rows = []
    for agent_name, evaluation in result_set["evaluations"].items():
        rows.append(
            {
                "agent": agent_name,
                "games": evaluation["game_count"],
                "players": evaluation["player_count"],
                "avg_score": round(evaluation["average_score"], 3),
                "median_score": round(evaluation["median_score"], 3),
                "min_score": evaluation["min_score"],
                "max_score": evaluation["max_score"],
                "avg_turns": round(evaluation["average_turn_count"], 3),
                "avg_hints": round(evaluation["average_hint_tokens"], 3),
                "avg_strikes": round(evaluation["average_strike_tokens"], 3),
                "avg_completed_stacks": round(evaluation["average_completed_stacks"], 3),
                "avg_play_actions": round(evaluation["average_play_actions"], 3),
                "avg_successful_plays": round(evaluation["average_successful_plays"], 3),
                "avg_failed_plays": round(evaluation["average_failed_plays"], 3),
                "avg_discards": round(evaluation["average_discards"], 3),
                "avg_hints_given": round(evaluation["average_hints_given"], 3),
                "successful_play_rate": round(evaluation["successful_play_rate"], 4),
                "win_rate": round(evaluation["win_rate"], 4),
                "loss_rate": round(evaluation["loss_rate"], 4),
                "score>=10": round(evaluation["score_at_least_10_rate"], 4),
                "score>=15": round(evaluation["score_at_least_15_rate"], 4),
            }
        )
    return rows


def safe_ratio(numerator, denominator):
    return round(numerator / denominator, 4) if denominator else None


def build_derived_rows(rows):
    return [
        {
            "agent": row["agent"],
            "score_per_hint": safe_ratio(row["avg_score"], row["avg_hints_given"]),
            "score_per_play": safe_ratio(row["avg_score"], row["avg_play_actions"]),
            "plays_per_hint": safe_ratio(row["avg_play_actions"], row["avg_hints_given"]),
            "discards_per_score": safe_ratio(row["avg_discards"], row["avg_score"]),
            "turns_per_score": safe_ratio(row["avg_turns"], row["avg_score"]),
        }
        for row in rows
    ]


def print_table(rows, title):
    print(title)
    headers = list(rows[0].keys())
    widths = {
        header: max(len(header), max(len(str(row[header])) for row in rows))
        for header in headers
    }

    header_line = " | ".join(header.ljust(widths[header]) for header in headers)
    separator_line = "-+-".join("-" * widths[header] for header in headers)

    print(header_line)
    print(separator_line)
    for row in rows:
        print(" | ".join(str(row[header]).ljust(widths[header]) for header in headers))
    print()


result_sets_by_player_count = {
    result_set["player_count"]: result_set for result_set in report["result_sets"]
}


In [ ]:
for player_count in report["config"]["player_counts"]:
    result_set = result_sets_by_player_count[player_count]
    rows = build_rows(result_set)
    derived_rows = build_derived_rows(rows)

    print_table(rows, f"=== {player_count}-Player Metrics ===")
    print_table(derived_rows, f"=== {player_count}-Player Derived Efficiency ===")

    print("Ranking")
    for index, entry in enumerate(result_set["ranking"], start=1):
        print(
            f"  {index}. {entry['agent']} | average_score={entry['average_score']:.3f} "
            f"| score_at_least_15_rate={entry['score_at_least_15_rate']:.3%} "
            f"| loss_rate={entry['loss_rate']:.3%}"
        )
    print()

    print("Comparisons")
    for label, comparison in result_set["comparisons"].items():
        print(
            f"  {label}: average_score_delta={comparison['average_score_delta']:+.3f}, "
            f"win_rate_delta={comparison['win_rate_delta']:+.3%}, "
            f"score_at_least_15_rate_delta={comparison['score_at_least_15_rate_delta']:+.3%}"
        )
    print()

    print("Score distributions")
    for agent_name, evaluation in result_set["evaluations"].items():
        print(f"  {agent_name}: {evaluation['score_distribution']}")
    print()

print("=== Aggregate Ranking Across Player Counts ===")
for index, entry in enumerate(report["aggregate_ranking"], start=1):
    print(f"  {index}. {entry['agent']} | average_score={entry['average_score']:.3f}")
